# Solution F2 · When boosting does not win: extrapolation, noisy linear signal, interactions

**Type**: Compare

## Objective

Compare Ridge, RandomForest and XGBoost on **five regimes** built to show the strengths and weaknesses of trees: a trend to extrapolate, a linear signal with a lot of noise (with `n = 150` and `n = 5000`), a signal made of interactions (with `n = 600` and `n = 5000`), and the hybrid remedy "Ridge + XGBoost on the residuals".

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error


def trend(seed=0):
    rng = np.random.default_rng(seed)
    n = 1200
    t = np.linspace(0, 10, n)
    X = np.c_[t, np.sin(3 * t)]
    y = 2 * t + np.sin(3 * t) + rng.normal(0, 0.5, n)
    c = int(0.7 * n)
    return X[:c], y[:c], X[c:], y[c:]


def noisy_linear(n, seed):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n + 2000, 30))
    beta = np.zeros(30)
    beta[:6] = [1, -1, 0.8, -0.8, 0.5, 0.5]
    y = X @ beta + rng.normal(0, 2, n + 2000)
    return X[:n], y[:n], X[n:], y[n:]


def interactions(n, seed):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n + 2000, 8))
    y = np.where(X[:, 0] > 0, X[:, 1], -X[:, 1]) * 2 + (X[:, 2] * X[:, 3] > 0) * 1.5 + rng.normal(0, 0.5, n + 2000)
    return X[:n], y[:n], X[n:], y[n:]


regimes = {
    "trend (extrapolation)": trend(),
    "noisy linear, n=150": noisy_linear(150, 1),
    "noisy linear, n=5000": noisy_linear(5000, 2),
    "interactions, n=600": interactions(600, 3),
    "interactions, n=5000": interactions(5000, 4),
}
models = {
    "ridge": lambda: RidgeCV(alphas=np.logspace(-2, 3, 20)),
    "rf": lambda: RandomForestRegressor(300, min_samples_leaf=3, random_state=0, n_jobs=-1),
    "xgb": lambda: xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=0, n_jobs=-1),
}
rmse = lambda y, p: mean_squared_error(y, p, squared=False)

## Commented solution

In [ ]:
table = pd.DataFrame(index=list(regimes), columns=list(models), dtype=float)
for regime, (X_tr, y_tr, X_te, y_te) in regimes.items():
    for name, create in models.items():
        table.loc[regime, name] = rmse(y_te, create().fit(X_tr, y_tr).predict(X_te))
winner = table.idxmin(axis=1)
xgb_ridge_ratio = table["xgb"] / table["ridge"]
print(table.round(3).to_string())
print("\nwinner:", winner.to_dict())
print("xgb / ridge:", xgb_ridge_ratio.round(2).to_dict())

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, y_tr, X_te, y_te = regimes["trend (extrapolation)"]
X_t, y_t = np.r_[X_tr, X_te], np.r_[y_tr, y_te]
A, B, ya, yb = train_test_split(X_t, y_t, test_size=0.3, random_state=0)
random_trend_table = pd.Series({name: rmse(yb, create().fit(A, ya).predict(B)) for name, create in models.items()})
random_trend_ratio = float(random_trend_table["xgb"] / random_trend_table["ridge"])
print("trend with a random split:", random_trend_table.round(3).to_dict(), "| xgb / ridge", round(random_trend_ratio, 2))

In [ ]:
def hybrid(X_tr, y_tr, X_te):
    linear = models["ridge"]().fit(X_tr, y_tr)
    rest = models["xgb"]().fit(X_tr, y_tr - linear.predict(X_tr))
    return linear.predict(X_te) + rest.predict(X_te)


hybrid_rmse = pd.Series({regime: rmse(y_te, hybrid(X_tr, y_tr, X_te)) for regime, (X_tr, y_tr, X_te, y_te) in regimes.items()})
print(pd.DataFrame({"best single": table.min(axis=1), "hybrid": hybrid_rmse}).round(3).to_string())

In [ ]:
rules = {
    "trend": "Linear model (or hybrid): trees do not extrapolate outside the training range and predict a constant.",
    "noisy linear": "Ridge/regularized: with a linear signal and a lot of noise, trees spend useless flexibility and overfit.",
    "interactions": "Boosting or forest: they capture the nonlinear interactions that a linear model cannot represent.",
}
rules

## Check

In [ ]:
with verify():
    check(table.notna().all().all() and table.shape == (5, 3), "complete 5 x 3 table", "RMSE per regime and method")
    check(winner is not None and winner["trend (extrapolation)"] == "ridge" and winner["noisy linear, n=150"] == "ridge" and winner["noisy linear, n=5000"] == "ridge", "Ridge wins on the trend and on the noisy linear signal (even with 5000 examples)", "idxmin per row")
    check(winner is not None and winner["interactions, n=600"] in ("rf", "xgb") and winner["interactions, n=5000"] in ("rf", "xgb") and table.loc["interactions, n=5000", "ridge"] > 2.5 * table.loc["interactions, n=5000", "xgb"], "with interactions the trees win, and Ridge has an RMSE over 2.5 times larger at n=5000", "ridge does not see the interactions")
    check(xgb_ridge_ratio is not None and xgb_ridge_ratio["trend (extrapolation)"] > 4 and 1.0 < xgb_ridge_ratio["noisy linear, n=5000"] < 1.15 and 1.02 < xgb_ridge_ratio["noisy linear, n=150"], "on the trend XGBoost has an RMSE over 4 times larger; on the noisy linear one it is always worse (slightly at n=5000, less than 15%)", "xgb / ridge")
    check(random_trend_table is not None and random_trend_ratio < 1.2 and random_trend_table.min() < 0.6, "with a random split the three methods are close (xgb / ridge below 1.2): the disaster was the extrapolation", "random 70/30 split of the trend regime")
    check(len(hybrid_rmse) == 5 and all(hybrid_rmse[r] <= 1.1 * table.loc[r].min() for r in table.index), "the hybrid is within 10% of the best in all the regimes", "ridge + xgb on the residuals")
    check(hybrid_rmse["trend (extrapolation)"] < 0.7 and hybrid_rmse["interactions, n=5000"] < 0.85, "the hybrid solves the trend (below 0.7) and the interactions at n=5000 (below 0.85)", "the trees correct only the residuals")
    check(set(rules) == {"trend", "noisy linear", "interactions"} and all(isinstance(v, str) and len(v) > 15 for v in rules.values()), "three rules, one per regime", "one sentence per regime")
done()

## What you learned

- **Trees do not extrapolate:** on the trend outside the training range XGBoost has an RMSE of 2.98 against 0.48 for Ridge (6 times), and the forest 3.30. The model predicts the mean of the outermost leaves, that is a constant. With a random split (interpolation) the difference disappears (0.54 against 0.51): the problem is not the function but being out of range. On data with a time trend, prices or growth, a tree alone is not enough.
- **With a noisy linear signal Ridge always wins,** even at n=5000 (1.98 against 2.04, 3%): with 150 examples the gap rises to 10% (2.24 against 2.47). Trees fragment the signal into thresholds and pay for its noise. Before installing XGBoost, a regularized linear baseline is a 10-second test.
- **With interactions the picture flips:** Ridge has RMSE 2.15 at n=5000 against 0.71 for XGBoost (3 times worse) because it cannot represent `±x₁` as a function of the sign of `x₀`. The advantage grows with `n`: at n=600 XGBoost is 1.16 and Ridge 2.13 (1.8 times); at n=5000 the ratio reaches 3.
- **The hybrid is safe:** a linear model captures the global structure (and the extrapolation), boosting on the residuals the nonlinear part. In all five regimes it is within 6% of the best single model (trend: 0.51 against 0.48; linear: 2.28 against 2.24 and 2.01 against 1.98) and with the interactions it is even better (1.08 against 1.16; 0.65 against 0.71). It is the configuration to try when you do not know which regime your problem belongs to.

## Common pitfall

Applying boosting "because it wins on Kaggle" to a problem with a time trend or one that has to be extrapolated: outside the range it sees a constant. And comparing only with other trees: the regularized linear baseline always goes in the table.

## Interview question

> *Click on a question to show its answer.*

<details><summary>When would you choose a linear model instead of boosting?</summary>

With a mostly linear and noisy signal, few data, the need to extrapolate outside the training range (trends, growth), or requirements of interpretability and stability. Boosting wins with interactions and nonlinearity and with enough data. In any case I always put a regularized linear baseline in the table and also evaluate the hybrid
(linear plus boosting on the residuals).

</details>